# Punjabi Audio Transcription from Google Drive

In [18]:
# Install necessary libraries
!pip install transformers torch peft librosa accelerate bitsandbytes

## Transcription Function

This function will load the fine-tuned Whisper model and transcribe your audio file.

In [19]:
import torch
from transformers import AutoModelForSpeechSeq2Seq, AutoProcessor
from peft import PeftModel
import librosa

def transcribe_punjabi_audio(audio_path, repo_id, base_model_id="openai/whisper-medium"):
    """
    Transcribes a local audio file using a fine-tuned Whisper model with LoRA adapters.

    Args:
        audio_path (str): Path to the local audio file (e.g., .mp3, .wav, .m4a).
        repo_id (str): Your Hugging Face repository ID containing the LoRA adapters.
        base_model_id (str): The underlying base architecture model.
    """
    # 1. Determine device setup
    device = "cuda" if torch.cuda.is_available() else "cpu"
    torch_dtype = torch.float16 if torch.cuda.is_available() else torch.float32

    print(f"Loading processor and base model: {base_model_id}...")

    # 2. Load the processor (combines feature extractor and tokenizer)
    try:
        processor = AutoProcessor.from_pretrained(repo_id, language="punjabi", task="transcribe")
    except Exception:
        processor = AutoProcessor.from_pretrained(base_model_id, language="punjabi", task="transcribe")

    # 3. Load base model with low-resource settings
    base_model = AutoModelForSpeechSeq2Seq.from_pretrained(
        base_model_id,
        torch_dtype=torch_dtype,
        low_cpu_mem_usage=True,
        device_map="auto" if device == "cuda" else None
    )

    print(f"Merging trained LoRA adapters from {repo_id}...")
    # 4. Wrap the base model with your custom LoRA adapter layers
    model = PeftModel.from_pretrained(base_model, repo_id)
    model.to(device)
    model.eval()

    print(f"Processing audio file: {audio_path}...")
    # 5. Load and resample the audio to 16,000 Hz (Crucial for Whisper)
    audio_array, sampling_rate = librosa.load(audio_path, sr=16000)

    # 6. Extract log-mel spectrogram features from the raw audio waveform
    input_features = processor(
        audio_array,
        sampling_rate=sampling_rate,
        return_tensors="pt"
    ).input_features.to(device).to(torch_dtype)

    print("Generating transcription...")
    # 7. Predict token IDs
    with torch.no_grad():
        predicted_ids = model.generate(
            input_features,
            language="punjabi",
            task="transcribe"
        )

    # 8. Decode the token IDs back into readable Gurmukhi script text
    transcription = processor.batch_decode(predicted_ids, skip_special_tokens=True)[0]

    return transcription

## Mount Google Drive

Run the cell below to mount your Google Drive. This will allow you to access files stored in your Drive directly from this Colab notebook.

In [11]:
from google.colab import drive
drive.mount('/content/drive')

print("Google Drive mounted successfully!")

Mounted at /content/drive
Google Drive mounted successfully!


## Transcribe Audio from Google Drive

**IMPORTANT**: Replace the `TARGET_AUDIO_FROM_DRIVE` variable with the *actual path* to your audio file within your mounted Google Drive. For example, if your file is in `MyDrive/audio/my_recording.m4a`, the path would be `/content/drive/MyDrive/audio/my_recording.m4a`.

In [21]:
# Replace this with the actual path to your audio file in Google Drive
TARGET_AUDIO_FROM_DRIVE = '/content/drive/MyDrive/AnnamAI Tasks/Outreach Activity STT + Question Generation Workflow/Sample Audio Files/MarauliKhurad2.m4a'

# Path where you want to save the transcription in Google Drive
SAVE_PATH_DRIVE = '/content/drive/MyDrive/AnnamAI Tasks/Outreach Activity STT + Question Generation Workflow/Sample Audio Files/Transcriptions-finetuned-medium-whisper/MarauliKhurad2.txt'

# Your Hugging Face repository ID
HF_REPO_ID = "KaliNangia/whisper-medium-gurmukhi-lora"

try:
    # Use the transcribe_punjabi_audio function defined earlier
    result = transcribe_punjabi_audio(audio_path=TARGET_AUDIO_FROM_DRIVE, repo_id=HF_REPO_ID)
    print("\n--- Transcription Result ---")
    print(result)

    # Create directory if it doesn't exist
    import os
    os.makedirs(os.path.dirname(SAVE_PATH_DRIVE), exist_ok=True)

    # Save the result directly to Google Drive
    with open(SAVE_PATH_DRIVE, "w", encoding="utf-8") as f:
        f.write(result)

    # Also save a local copy for convenience
    with open("transcription_from_drive.txt", "w", encoding="utf-8") as f:
        f.write(result)

    print(f"\n✅ Saved to Google Drive at: {SAVE_PATH_DRIVE}")
    print("💾 Also saved locally as 'transcription_from_drive.txt'!")
except FileNotFoundError:
    print(f"Error: Could not find audio file at '{TARGET_AUDIO_FROM_DRIVE}'.")
except Exception as e:
    print(f"An error occurred: {e}")

Loading processor and base model: openai/whisper-medium...


Loading weights:   0%|          | 0/947 [00:00<?, ?it/s]

Merging trained LoRA adapters from KaliNangia/whisper-medium-gurmukhi-lora...
Processing audio file: /content/drive/MyDrive/AnnamAI Tasks/Outreach Activity STT + Question Generation Workflow/Sample Audio Files/MarauliKhurad2.m4a...


/tmp/ipykernel_576/2355315845.py:43: UserWarning: PySoundFile failed. Trying audioread instead.
  audio_array, sampling_rate = librosa.load(audio_path, sr=16000)
/usr/local/lib/python3.12/dist-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)


Generating transcription...

--- Transcription Result ---
ਕਾ ਸ਼ਿਕਾਲ ਜੀ ਆਪਣਾ ਨਾਮ ਜੀ ਕਿਦੇ ਤਾਰ ਸੇਂ ਤੇ ਪੇਂਡੇ ਸਾ ਰਾਪਣਾ ਮੜੋਡੀ ਕੋਰਦੇ ਮੜੋਡੀ ਕੋਰਦੇ ਪੱਪੂ ਜੇ ਪਹ ਕਿੰਿੰਂ ਕਿਹਿੀ ਕਿੱਤੀ ਗਰਦੇ ਜੀ ਜੈਫੋਂ ਤਾ ਦੇਖੋਂ ਹੁਣਿਗਾਰ ਕੋਇ ਕੈੱਂ ਕੋਂ ਕੈੱਂ ਕੈੱਂ ਕੈੱਂ ਕੈੱਂ ਕੈੱਂ ਕੈੱਂ ਕੈੱਂ ਕੈੱਂ ਕੈੱਂ ਕੈੱਂ ਕੈੱਂ ਕੈੱਂ ਕ

✅ Saved to Google Drive at: /content/drive/MyDrive/AnnamAI Tasks/Outreach Activity STT + Question Generation Workflow/Sample Audio Files/Transcriptions-finetuned-medium-whisper/MarauliKhurad2.txt
💾 Also saved locally as 'transcription_from_drive.txt'!
